In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

In [ ]:
OWNER = "OWNER"  # tools/kaggle_ops.py --owner replaces this before pushing
# On Kaggle the inputs live under /kaggle/input and outputs under /kaggle/working; tests point these at a fake tree
KAGGLE_INPUT = Path(os.environ.get("KAGGLE_INPUT", "/kaggle/input"))
WORKING = Path(os.environ.get("KAGGLE_WORKING", "/kaggle/working"))
COMP_DIR = KAGGLE_INPUT / "competitions" / "rsna-knee-abnormality-detection"
PKG_DIR = KAGGLE_INPUT / "datasets" / OWNER / "rsna-knee-8th-pkg"
MODEL_ROOT = KAGGLE_INPUT / "datasets" / OWNER / "rsna-knee-8th-model"
OUT = WORKING / "submission.csv"
# Experiments to score. Each outer element is one part; experiments inside a part are averaged as probabilities
# PART_WEIGHTS: one weight per part (a number or 12 per-finding weights). None averages all members equally
# BLEND: how parts are mixed ("rank" per-finding ranks or "prob" probabilities)
# Submission B (57523282): 4 parts, 0.25 each, probability blend (ctxenc x4, slotctx x4, CAFormer-S36 110 mm x3, CAFormer-S36 140 mm x4)
Ensemble = [["coatnet_spatial_lsewin03_s2online_union_ctxattn_headreset_ctxenc_ep10_aug_mixlabel_v6t275r040_pseudo2v6r040_cos20_softlse_half_all_seriesaug_ema_usub_dprior_a10"],
            ["coatnet_spatial_lsewin03_s2online_union_ctxattn_slotctx_headreset_ep10_aug_mixlabel_v6t275r040_pseudo2v6r040_knees_cos20_softlse_half_all_seriesaug_ema_usub_dprior_a10"],
            ["caformers36_spatial_planeaux01_c110a10_s2online_union_seriesneck_headreset_ep10_aug_v6r2r040_knees_cos20_softlse_half_gold_seriesaug_ema_usub_dprior_a10"],
            ["caformers36_spatial_planeaux01_s2online_union_seriesneck_headreset_ep10_aug_v6r2r040_knees_cos20_softlse_half_gold_seriesaug_ema_usub_dprior_a10"]]
PART_WEIGHTS = [0.25, 0.25, 0.25, 0.25]
BLEND = "prob"
WORKERS = 3  # DICOM decoding worker processes (4 vCPUs)
PREFETCH = 6  # studies decoded ahead of the GPU
IO_THREADS = 8  # DICOM reader threads per worker (hides NFS latency)

In [ ]:
sys.path.insert(0, str(PKG_DIR))
import rsna_knee.data.dicom as dicom_mod  # noqa: E402
from rsna_knee.constants import ID_COLUMN, LABELS  # noqa: E402
from rsna_knee.pipeline import iter_studies, place_models, score_stream  # noqa: E402
from rsna_knee.predict import load_experiments, rank_average, resolve_devices  # noqa: E402

dicom_mod.IO_THREADS = IO_THREADS

In [ ]:
# Experiments are mixed equally and folds share each experiment's weight equally (members.weights)
if PART_WEIGHTS is not None and len(PART_WEIGHTS) != len(Ensemble):
    raise ValueError(f"PART_WEIGHTS has {len(PART_WEIGHTS)} weights for {len(Ensemble)} parts")
if BLEND not in ("rank", "prob"):
    raise ValueError(f"BLEND must be 'rank' or 'prob', got {BLEND!r}")
torch.set_num_threads(1)  # Leave the CPUs to the workers. Do not touch CUDA yet (the next cell forks)
members = load_experiments(MODEL_ROOT, Ensemble)

In [ ]:
test = pd.read_csv(COMP_DIR / "test.csv", dtype={ID_COLUMN: str})
series = pd.read_csv(COMP_DIR / "test_series.csv", dtype={ID_COLUMN: str})
series_rows = {uid: frame.to_dict(orient="records") for uid, frame in series.groupby(ID_COLUMN, sort=False)}
items = [(COMP_DIR / "test_series" / study_id, series_rows.get(study_id, [])) for study_id in test[ID_COLUMN]]
# Fork the decoding workers here, before CUDA is initialised
stream = iter_studies(items, members.group_params, workers=WORKERS, prefetch=PREFETCH)

In [ ]:
banks = place_models(members.models, resolve_devices("cuda"))  # Every member on every GPU


def decoded():
    """Hand studies from the workers to the consumer as (pixels, mask, ...); decoding errors are raised."""
    for index, studies in stream:
        if isinstance(studies, Exception):
            raise studies
        yield index, [(study.pixels, study.mask, study.series_headers, study.subseries) for study in studies]


probs = [None] * len(test)
for index, result in score_stream(decoded(), banks, members.group_members, weights=members.weights, parts=members.parts if PART_WEIGHTS is not None else None):
    if isinstance(result, Exception):
        raise result
    probs[index] = result
stream.close()

In [ ]:
if PART_WEIGHTS is None:
    blended = np.vstack(probs)
else:
    stacked = np.stack(probs)  # [study, part, finding]
    if BLEND == "rank":
        scores = np.stack([rank_average(stacked[:, [part]]) for part in range(stacked.shape[1])])  # [part, study, finding]
    else:
        # members.weights sum to 1 over all members, so divide by each part's total to get back that part's probability
        totals = np.zeros((len(Ensemble), len(LABELS)))
        for weight, part in zip(members.weights, members.parts):
            totals[part] += np.broadcast_to(np.asarray(weight, dtype=float), len(LABELS))
        scores = stacked.transpose(1, 0, 2) / totals[:, None, :]  # [part, study, finding]
    weights = np.array([np.broadcast_to(np.asarray(weight, dtype=float), len(LABELS)) for weight in PART_WEIGHTS])  # [part, finding]
    blended = (weights[:, None, :] * scores).sum(axis=0) / weights.sum(axis=0)
submission = pd.DataFrame(blended, columns=LABELS)
submission.insert(0, ID_COLUMN, test[ID_COLUMN].to_numpy())
submission.to_csv(OUT, index=False)